# Concurrency

Python has three distinct concurrency models, each suited to a different problem. Threading handles I/O-bound work where threads spend most of their time waiting. `asyncio` does the same but with a single thread and cooperative scheduling; thousands of concurrent tasks, no overhead. Multiprocessing breaks through the GIL for CPU-bound work by running real parallel processes.

**What's inside:** `ThreadPoolExecutor` for parallel I/O, `asyncio` with `async`/`await` and `gather`, `ProcessPoolExecutor` for CPU-bound parallelism, and a guide to choosing the right model.

**Learn more:** [concurrent.futures](https://docs.python.org/3/library/concurrent.futures.html) · [asyncio](https://docs.python.org/3/library/asyncio.html) · [multiprocessing](https://docs.python.org/3/library/multiprocessing.html)

## 1. Threading: I/O-bound parallelism

`ThreadPoolExecutor` runs tasks in a thread pool. Threads share memory and release the GIL during I/O, making this ideal for network calls, file reads, and anything that waits.

### 1.1 submit: fire and collect results

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch(url):
    time.sleep(0.1)          # simulate a network call
    return f'got {url}'

urls = [f'https://example.com/page/{i}' for i in range(10)]

start = time.perf_counter()

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = [pool.submit(fetch, url) for url in urls]
    results = [f.result() for f in futures]

elapsed = time.perf_counter() - start
print(f'{len(results)} results in {elapsed:.2f}s  (sequential would take ~{len(urls)*0.1:.1f}s)')
print(results[:3])

### 1.2 map: simpler interface for uniform tasks

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def process(n):
    time.sleep(0.05)
    return n ** 2

with ThreadPoolExecutor(max_workers=4) as pool:
    # map preserves order and returns an iterator
    squares = list(pool.map(process, range(8)))

print(squares)

### 1.3 as_completed: process results as they arrive

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

def slow_task(n):
    time.sleep(random.uniform(0.05, 0.2))
    return n * 10

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(slow_task, i): i for i in range(6)}
    for future in as_completed(futures):
        task_id = futures[future]
        print(f'task {task_id} → {future.result()}')

## 2. asyncio: cooperative concurrency

`asyncio` uses a single thread and a cooperative scheduler. Coroutines (`async def`) voluntarily yield at `await` points, letting other coroutines run. No threads, no GIL concerns, and it scales to thousands of concurrent tasks.

> **Jupyter note:** Jupyter already runs an event loop, so use `await` directly at the top level (`asyncio.run()` will raise a `RuntimeError` here).

### 2.1 async def and await

In [ ]:
import asyncio

async def fetch(url, delay=0.1):
    await asyncio.sleep(delay)   # cooperative: yields control while waiting
    return f'got {url}'

# in Jupyter, top-level await works directly
result = await fetch('https://example.com')
print(result)

### 2.2 gather: run coroutines concurrently

In [ ]:
import asyncio, time

async def fetch(url, delay=0.1):
    await asyncio.sleep(delay)
    return f'got {url}'

urls = [f'https://example.com/page/{i}' for i in range(10)]

start = time.perf_counter()
# gather runs all coroutines concurrently and collects results in order
results = await asyncio.gather(*[fetch(url) for url in urls])
elapsed = time.perf_counter() - start

print(f'{len(results)} results in {elapsed:.2f}s  (sequential would take ~{len(urls)*0.1:.1f}s)')
print(results[:3])

### 2.3 Tasks and cancellation

In [ ]:
import asyncio

async def worker(name, delay):
    await asyncio.sleep(delay)
    return f'{name} done'

# create_task schedules coroutines independently
task_a = asyncio.create_task(worker('A', 0.1))
task_b = asyncio.create_task(worker('B', 0.05))
task_c = asyncio.create_task(worker('C', 0.15))

results = await asyncio.gather(task_a, task_b, task_c)
print(results)

### 2.4 asyncio.timeout: don't wait forever

In [ ]:
import asyncio

async def slow_operation():
    await asyncio.sleep(5)
    return 'done'

try:
    async with asyncio.timeout(0.2):    # Python 3.11+
        result = await slow_operation()
except asyncio.TimeoutError:
    print('timed out after 0.2s')

## 3. Multiprocessing: CPU-bound parallelism

The GIL prevents true parallel CPU execution on threads. `ProcessPoolExecutor` spawns separate Python processes, each with its own GIL, so CPU-heavy work runs in parallel.

### 3.1 ProcessPoolExecutor

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time

def cpu_intensive(n):
    # simulate heavy computation
    return sum(i * i for i in range(n))

tasks = [500_000] * 8

# sequential
start = time.perf_counter()
seq_results = [cpu_intensive(n) for n in tasks]
seq_time = time.perf_counter() - start

# parallel (uses all CPU cores)
start = time.perf_counter()
with ProcessPoolExecutor() as pool:
    par_results = list(pool.map(cpu_intensive, tasks))
par_time = time.perf_counter() - start

print(f'sequential: {seq_time:.2f}s')
print(f'parallel:   {par_time:.2f}s')
print(f'speedup:    {seq_time / par_time:.1f}x')

## 4. Choosing the Right Model

| Scenario | Best Tool | Why |
|---|---|---|
| I/O-bound, many tasks, simple | `ThreadPoolExecutor` | Low overhead, familiar API |
| I/O-bound, very many tasks | `asyncio` | Single thread, scales to 10k+ concurrent ops |
| CPU-bound, data parallel | `ProcessPoolExecutor` | Bypasses GIL, uses all cores |
| Mixed I/O + CPU | `asyncio` + `ProcessPoolExecutor` | Combine both models |

In [ ]:
# combining asyncio with ProcessPoolExecutor for CPU work inside an async app
import asyncio
from concurrent.futures import ProcessPoolExecutor

def heavy(n):
    return sum(i * i for i in range(n))

async def main():
    loop = asyncio.get_event_loop()
    with ProcessPoolExecutor() as pool:
        # run_in_executor bridges sync code into an async context
        results = await asyncio.gather(
            loop.run_in_executor(pool, heavy, 200_000),
            loop.run_in_executor(pool, heavy, 300_000),
            loop.run_in_executor(pool, heavy, 400_000),
        )
    return results

results = await main()
print(results)